# Preprocesado de datos y división del dataset

En este notebook vamos a preparar los datos para el entrenamiento del modelo clasificador de riesgo IA (AI Act).

Los pasos que seguiremos son:
1. Carga del dataset limpio
2. Aplicar NER (Named Entity Recognition) a la columna `descripcion`
3. Explorar las entidades extraídas por tipo y clase de riesgo
4. Preprocesar el texto con lematización
5. Dividir en train / validation / test (70% / 15% / 15%) con estratificación
6. Guardar los conjuntos en `data/processed/`

In [ ]:
#Al principio de cada notebook, añadimos estas líneas para que se recarguen automáticamente las funciones que hayamos editado en el módulo functions.py sin tener que reiniciar el kernel cada vez.
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

path = "data/dataset_riesgo_limpio.csv"
df = pd.read_csv(path)

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")
print(f"\nDistribución de clases:")
print(df["etiqueta"].value_counts())

## 1. Aplicar NER al dataset

In [ ]:
from functions import extraer_entidades, resumen_entidades

# Aplicamos NER a la columna de descripción original (sin limpiar)
df = extraer_entidades(df, "descripcion")

# Mostramos las entidades de las primeras 3 filas como ejemplo
for i in range(3):
    print(f"Texto: {df['descripcion'].iloc[i][:100]}...")
    print(f"Entidades: {df['entidades'].iloc[i]}")
    print()

In [ ]:
# Resumen de entidades por tipo y por clase de riesgo
freq_por_tipo, freq_por_tipo_clase = resumen_entidades(df)

### Conclusión del NER

Las entidades nombradas (NER) en este dataset regulatorio no aportan señal discriminativa significativa para la clasificación de riesgo. Los textos describen sistemas genéricos de IA sin mencionar nombres propios, organizaciones o ubicaciones específicas que diferencien las clases.

Por tanto, **continuamos el entrenamiento utilizando solo el texto limpio y lematizado** como feature principal.

## 2. Preprocesado del texto con lematización

In [ ]:
from functions import preparar_dataset

# preparar_dataset ahora usa limpiar_texto_preprocess (con lematización)
df_processed = preparar_dataset(df, "descripcion", "etiqueta")

print(f"Dataset procesado: {df_processed.shape}")
print(f"\nEjemplo de texto lematizado:")
print(f"  Original:   {df['descripcion'].iloc[0][:100]}...")
print(f"  Lematizado: {df_processed['text_final'].iloc[0][:100]}...")

## 3. División en train / validation / test

In [ ]:
from functions import split_dataset

# Dividimos en train (70%), validation (15%) y test (15%) con estratificación
train_df, val_df, test_df = split_dataset(df_processed, "etiqueta")

print(f"\nDistribución por clase en cada conjunto:")
print("\nTrain:")
print(train_df["etiqueta"].value_counts())
print("\nValidation:")
print(val_df["etiqueta"].value_counts())
print("\nTest:")
print(test_df["etiqueta"].value_counts())

## 4. Guardar los conjuntos procesados

In [ ]:
import os

output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

train_df.to_csv(os.path.join(output_dir, "train.csv"), index=False)
val_df.to_csv(os.path.join(output_dir, "validation.csv"), index=False)
test_df.to_csv(os.path.join(output_dir, "test.csv"), index=False)

print(f"Archivos guardados en {output_dir}/:")
for f in os.listdir(output_dir):
    print(f"  {f}")